# 05 · Reverse-phi and neural analyses
Reverse-phi illusion plus recurrence, locality and representational similarity.

In [ ]:
import torch, numpy as np
from flynet_robustness.models import get_flynet, forward_flynet
from flynet_robustness.decoders import load_decoder
from flynet_robustness.hex_eye import BoxEye
from flynet_robustness.rendering import generate_looming_stimulus
from flynet_robustness.textures import get_texture
from flynet_robustness.reverse_phi import reverse_phi_p_hit
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('FlyNet', device=device)
decoder = load_decoder('FlyNet')
renderer_hex = BoxEye(model1.connectome.config.extent, kernel_size=13)

In [ ]:
vis, *_ = generate_looming_stimulus(hit=True, radius=4.0, shape='Sphere',
    texture=get_texture('Black'), time=1.0, device=device, z0=15, out_size=416)
p_hit = reverse_phi_p_hit(vis, renderer_hex, decoder,
    forward_fn=lambda h: forward_flynet(h, model1, idxs1), T_flip=33)
from flynet_robustness.plotting import plot_p_hit_trace
plot_p_hit_trace(p_hit, color='crimson');

## Recurrence and connectivity

In [ ]:
from flynet_robustness.analysis.recurrence import (weight_matrix_from_network,
    recurrence, recurrence_weighted, recurrence_metrics)
W = weight_matrix_from_network(model1, signed=True)
print(recurrence_metrics(W))
print(recurrence(W, n_samples=500)[1])
print(recurrence_weighted(W, n_samples=500))

## Representational similarity

In [ ]:
from flynet_robustness.analysis.representational import trial_similarity, fit_global_pca
# activity: list of (T, N) arrays across trials
# population, neuron = trial_similarity(activity)
# pca = fit_global_pca(activity_all, n_components=3)